# Stock Price Direction Prediction

End-to-end ML pipeline using **live data** from Yahoo Finance API.


## Cell 1 — Imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, sys
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import yfinance as yf
import ta
from datetime import datetime

from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, f1_score, accuracy_score)
from xgboost import XGBClassifier
import shap
import joblib

sys.path.insert(0, '../src')
from data_fetcher import fetch_stock_data, get_latest_price
from feature_engineer import engineer_features
from trainer import prepare_data, train_and_evaluate, tune_best_model, save_artifacts

os.makedirs('../images', exist_ok=True)
os.makedirs('../models', exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("All libraries loaded successfully.")


All libraries loaded successfully.


## Cell 2 — Data Fetching (LIVE API CALL)

In [2]:
df = fetch_stock_data("AAPL", period="2y")

print(f"\nShape: {df.shape}")
print(f"\nHead:")
print(df.head())
print(f"\nTail:")
print(df.tail())
print(f"\n*** Data fetched LIVE from Yahoo Finance API at {datetime.now()} ***")



[*********************100%***********************]  1 of 1 completed

Ticker: AAPL
Date range: 2024-04-22 00:00:00 to 2026-04-20 00:00:00
Rows: 500

Shape: (500, 6)

Head:
        Date       Close        High         Low        Open    Volume
0 2024-04-22  164.344299  165.751491  163.283957  164.027193  48116400
1 2024-04-23  165.394745  165.543401  163.432606  163.858736  49537800
2 2024-04-24  167.495651  167.773125  164.710996  165.038007  48251800
3 2024-04-25  168.357819  169.071326  166.633506  168.001065  50558300
4 2024-04-26  167.773102  169.794697  167.654174  168.347873  44838400

Tail:
          Date       Close        High         Low        Open    Volume
495 2026-04-14  258.829987  261.929993  257.190002  259.250000  48370700
496 2026-04-15  266.429993  266.559998  257.809998  258.160004  49913500
497 2026-04-16  263.399994  267.160004  261.269989  266.799988  43323100
498 2026-04-17  270.230011  272.299988  266.720001  266.959991  61314800
499 2026-04-20  273.002991  274.274994  270.290009  270.329987  22160762

*** Data fetched LIVE from

## Cell 3 — Stock Price Visualisation

In [3]:
# Plot 1 — stock_price_history.png
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df['Date'], df['Close'], label='Close Price', color='#2c3e50', linewidth=1.5)
ax.plot(df['Date'], df['Close'].rolling(20).mean(), label='20-day SMA', color='#e74c3c', linewidth=1)
ax.plot(df['Date'], df['Close'].rolling(50).mean(), label='50-day SMA', color='#3498db', linewidth=1)
ax.set_title('AAPL — Stock Price History with Moving Averages')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend()
plt.tight_layout()
plt.savefig('../images/stock_price_history.png', bbox_inches='tight')
plt.show()
print("Insight: Moving average crossovers often signal trend changes — when the 20-day SMA crosses above the 50-day, it's a bullish signal.")


Insight: Moving average crossovers often signal trend changes — when the 20-day SMA crosses above the 50-day, it's a bullish signal.


In [4]:
# Plot 2 — volume_history.png
colors = ['green' if df['Close'].iloc[i] >= df['Open'].iloc[i] else 'red' for i in range(len(df))]
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(df['Date'], df['Volume'], color=colors, alpha=0.7, width=1)
ax.set_title('AAPL — Daily Trading Volume')
ax.set_xlabel('Date')
ax.set_ylabel('Volume')
plt.tight_layout()
plt.savefig('../images/volume_history.png', bbox_inches='tight')
plt.show()
print("Insight: Volume spikes often accompany major price moves — high volume confirms trend strength.")


Insight: Volume spikes often accompany major price moves — high volume confirms trend strength.


In [5]:
# Plot 3 — daily_returns_distribution.png
daily_ret = df['Close'].pct_change().dropna()
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(daily_ret, bins=50, alpha=0.6, color='#3498db', edgecolor='white', density=True)
sns.kdeplot(daily_ret, ax=ax, color='#e74c3c', linewidth=2)
ax.axvline(daily_ret.mean(), color='green', linestyle='--', label=f'Mean: {daily_ret.mean():.4f}')
ax.axvline(daily_ret.mean() + 2*daily_ret.std(), color='orange', linestyle='--', label=f'+2σ: {daily_ret.mean()+2*daily_ret.std():.4f}')
ax.axvline(daily_ret.mean() - 2*daily_ret.std(), color='orange', linestyle='--', label=f'-2σ: {daily_ret.mean()-2*daily_ret.std():.4f}')
ax.set_title('AAPL — Daily Returns Distribution')
ax.set_xlabel('Daily Return')
ax.legend()
plt.tight_layout()
plt.savefig('../images/daily_returns_distribution.png', bbox_inches='tight')
plt.show()
print(f"Insight: Returns are roughly normally distributed with mean {daily_ret.mean():.4f} and std {daily_ret.std():.4f}. Extreme moves beyond ±2σ are rare but impactful.")


Insight: Returns are roughly normally distributed with mean 0.0012 and std 0.0179. Extreme moves beyond ±2σ are rare but impactful.


## Cell 4 — Feature Engineering

In [6]:
data, feature_cols = engineer_features(df)

print(f"\nFeature columns ({len(feature_cols)}):")
for f in feature_cols:
    print(f"  - {f}")


Shape: (450, 29)
Features engineered: 22
Target distribution:
target
1    245
0    205
Name: count, dtype: int64

Feature columns (22):
  - returns_1d
  - returns_5d
  - returns_10d
  - log_return
  - sma_5
  - sma_20
  - sma_50
  - ema_12
  - ema_26
  - sma_cross
  - rsi_14
  - macd
  - macd_signal
  - macd_diff
  - stoch_k
  - bb_high
  - bb_low
  - bb_width
  - atr_14
  - volume_sma_20
  - volume_ratio
  - obv


In [7]:
# Plot 4 — feature_correlation.png
corr_with_target = data[feature_cols + ['target']].corr()['target'].drop('target').abs().sort_values(ascending=False)
top15 = corr_with_target.head(15)

fig, ax = plt.subplots(figsize=(10, 8))
top15_cols = list(top15.index) + ['target']
sns.heatmap(data[top15_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Top 15 Features — Correlation with Target')
plt.tight_layout()
plt.savefig('../images/feature_correlation.png', bbox_inches='tight')
plt.show()
print("Insight: The features most correlated with tomorrow's direction help us understand which technical signals the model will rely on.")


Insight: The features most correlated with tomorrow's direction help us understand which technical signals the model will rely on.


## Cell 5 — Technical Indicators Visualisation

In [8]:
# Plot 5 — technical_indicators.png (last 120 days)
recent = data.tail(120).copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Row 1: Price + Bollinger Bands
axes[0].plot(recent['Date'], recent['Close'], label='Close', color='#2c3e50')
axes[0].fill_between(recent['Date'], recent['bb_high'], recent['bb_low'], alpha=0.2, color='#3498db', label='Bollinger Bands')
axes[0].set_title('Price with Bollinger Bands')
axes[0].set_ylabel('Price ($)')
axes[0].legend(loc='upper left')

# Row 2: RSI
axes[1].plot(recent['Date'], recent['rsi_14'], color='#8e44ad', linewidth=1.5)
axes[1].axhline(70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
axes[1].fill_between(recent['Date'], 30, 70, alpha=0.1, color='gray')
axes[1].set_title('RSI (14)')
axes[1].set_ylabel('RSI')
axes[1].legend(loc='upper left')

# Row 3: MACD
axes[2].plot(recent['Date'], recent['macd'], label='MACD', color='#2980b9')
axes[2].plot(recent['Date'], recent['macd_signal'], label='Signal', color='#e74c3c')
axes[2].bar(recent['Date'], recent['macd_diff'], alpha=0.4, color='gray', label='Histogram')
axes[2].set_title('MACD')
axes[2].set_ylabel('MACD')
axes[2].legend(loc='upper left')

plt.tight_layout()
plt.savefig('../images/technical_indicators.png', bbox_inches='tight')
plt.show()
print("Insight: RSI above 70 signals overbought (potential reversal down), MACD crossing above signal is bullish.")


Insight: RSI above 70 signals overbought (potential reversal down), MACD crossing above signal is bullish.


## Cell 6 — Data Preparation (Time-Based Split)

In [9]:
X_train, X_test, y_train, y_test, scaler, fnames = prepare_data(data, feature_cols)


Train size: 360 | Test size: 90
TRAIN: 2024-07-02 to 2025-12-05
TEST:  2025-12-08 to 2026-04-17
No future data leaks into training — time-based split enforced.
Train target:
target
1    200
0    160
Name: count, dtype: int64
Test target:
target
0    45
1    45
Name: count, dtype: int64


## Cell 7 — Model Training & Comparison

In [10]:
results = train_and_evaluate(X_train, y_train, X_test, y_test)


Training: Logistic Regression


Accuracy : 0.5556
F1-Score : 0.6000
ROC-AUC  : 0.5363
              precision    recall  f1-score   support

        Down       0.57      0.44      0.50        45
          Up       0.55      0.67      0.60        45

    accuracy                           0.56        90
   macro avg       0.56      0.56      0.55        90
weighted avg       0.56      0.56      0.55        90

Training: Random Forest


Accuracy : 0.5222
F1-Score : 0.5057
ROC-AUC  : 0.5202
              precision    recall  f1-score   support

        Down       0.52      0.56      0.54        45
          Up       0.52      0.49      0.51        45

    accuracy                           0.52        90
   macro avg       0.52      0.52      0.52        90
weighted avg       0.52      0.52      0.52        90

Training: XGBoost


Accuracy : 0.4667
F1-Score : 0.4667
ROC-AUC  : 0.5170
              precision    recall  f1-score   support

        Down       0.47      0.47      0.47        45
          Up       0.47      0.47      0.47        45

    accuracy                           0.47        90
   macro avg       0.47      0.47      0.47        90
weighted avg       0.47      0.47      0.47        90



In [11]:
# model_comparison.png
metrics_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [r['acc'] for r in results.values()],
    'F1 Score': [r['f1'] for r in results.values()],
    'ROC-AUC': [r['auc'] for r in results.values()],
})

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(metrics_df))
width = 0.25
for i, m in enumerate(['Accuracy', 'F1 Score', 'ROC-AUC']):
    ax.bar(x + i*width, metrics_df[m], width, label=m)
ax.set_xticks(x + width)
ax.set_xticklabels(metrics_df['Model'])
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../images/model_comparison.png', bbox_inches='tight')
plt.show()


In [12]:
# roc_curves.png
fig, ax = plt.subplots(figsize=(9, 7))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    ax.plot(fpr, tpr, label=f"{name} (AUC = {r['auc']:.3f})")
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../images/roc_curves.png', bbox_inches='tight')
plt.show()


## Cell 8 — Hyperparameter Tuning (XGBoost)

In [13]:
best_model = tune_best_model(X_train, y_train, X_test, y_test)
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]


Fitting 5 folds for each of 108 candidates, totalling 540 fits


Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100, 'subsample': 1.0}
Best CV ROC-AUC: 0.5199

Tuned XGBoost on test set:
ROC-AUC : 0.4489
F1-Score: 0.4000
Accuracy: 0.5000
              precision    recall  f1-score   support

        Down       0.50      0.67      0.57        45
          Up       0.50      0.33      0.40        45

    accuracy                           0.50        90
   macro avg       0.50      0.50      0.49        90
weighted avg       0.50      0.50      0.49        90



## Cell 9 — Detailed Evaluation

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

prec, rec, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(rec, prec, color='#3498db', linewidth=2)
axes[1].fill_between(rec, prec, alpha=0.3, color='#3498db')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.savefig('../images/evaluation_detailed.png', bbox_inches='tight')
plt.show()


## Cell 10 — SHAP Explainability

In [15]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

plt.figure()
shap.summary_plot(shap_values, X_test, plot_type='bar', max_display=15, show=False)
plt.tight_layout()
plt.savefig('../images/shap_feature_importance.png', bbox_inches='tight')
plt.show()

plt.figure()
shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.tight_layout()
plt.savefig('../images/shap_beeswarm.png', bbox_inches='tight')
plt.show()

idx = int(np.argmax(y_proba))
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = float(np.array(expected_value).flatten()[0])

explanation = shap.Explanation(
    values=shap_values[idx],
    base_values=expected_value,
    data=X_test.iloc[idx].values,
    feature_names=list(X_test.columns)
)
plt.figure()
shap.plots.waterfall(explanation, max_display=10, show=False)
plt.tight_layout()
plt.savefig('../images/shap_waterfall.png', bbox_inches='tight')
plt.show()

print(f"Highest UP-probability sample index: {idx}")
print(f"Predicted UP probability: {y_proba[idx]:.4f}")
print(f"Actual label: {'Up' if y_test.iloc[idx]==1 else 'Down'}")


Highest UP-probability sample index: 37
Predicted UP probability: 0.6258
Actual label: Down


## Cell 11 — Save Artifacts

In [16]:
save_artifacts(best_model, scaler, fnames, path='../models/')


Saved: ../models/stock_model.pkl
Saved: ../models/scaler.pkl
Saved: ../models/feature_names.pkl


## Cell 12 — Live Prediction Demo

In [17]:
latest_df = fetch_stock_data("AAPL", period="2y")
latest_data, latest_fcols = engineer_features(latest_df)

latest_row = latest_data[latest_fcols].iloc[[-1]]
latest_scaled = pd.DataFrame(
    scaler.transform(latest_row), columns=latest_fcols, index=latest_row.index
)
live_proba = best_model.predict_proba(latest_scaled)[0, 1]
live_pred = "UP" if live_proba >= 0.5 else "DOWN"
live_close = float(latest_df['Close'].iloc[-1])

print("=" * 50)
print(f"LIVE PREDICTION ({datetime.now().strftime('%Y-%m-%d %H:%M')})")
print("=" * 50)
print(f"Stock       : AAPL")
print(f"Latest Close: ${live_close:.2f}")
print(f"Predicted   : {live_pred}")
print(f"Confidence  : {max(live_proba, 1-live_proba)*100:.1f}%")
print(f"\nThis prediction was made using data fetched LIVE from Yahoo Finance.")



[*********************100%***********************]  1 of 1 completed

Ticker: AAPL
Date range: 2024-04-22 00:00:00 to 2026-04-20 00:00:00
Rows: 500
Shape: (450, 29)
Features engineered: 22
Target distribution:
target
1    245
0    205
Name: count, dtype: int64
LIVE PREDICTION (2026-04-20 19:47)
Stock       : AAPL
Latest Close: $273.08
Predicted   : UP
Confidence  : 58.0%

This prediction was made using data fetched LIVE from Yahoo Finance.


## Cell 13 — Notebook Summary

In [18]:
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(mean_abs_shap)[::-1][:5]
top_features = [(X_test.columns[i], float(mean_abs_shap[i])) for i in top_idx]

img_count = len([f for f in os.listdir('../images') if f.endswith('.png')])
model_count = len([f for f in os.listdir('../models') if f.endswith('.pkl')])

print("=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)
print(f"Data source      : Yahoo Finance API (LIVE)")
print(f"Ticker           : AAPL")
print(f"Date range       : {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"Features         : {len(feature_cols)}")
print(f"Best model       : XGBoost (tuned)")
print(f"Test ROC-AUC     : {roc_auc_score(y_test, y_proba):.4f}")
print(f"Test F1 Score    : {f1_score(y_test, y_pred):.4f}")
print(f"Test Accuracy    : {accuracy_score(y_test, y_pred):.4f}")
print()
print("Top 5 features (by mean |SHAP|):")
for name, val in top_features:
    print(f"  - {name:<25} {val:.4f}")
print()
print(f"Images saved      : {img_count}")
print(f"Model artifacts   : {model_count}")
print(f"Timestamp         : {datetime.now()}")
print("=" * 60)


PROJECT SUMMARY
Data source      : Yahoo Finance API (LIVE)
Ticker           : AAPL
Date range       : 2024-04-22 to 2026-04-20
Features         : 22
Best model       : XGBoost (tuned)
Test ROC-AUC     : 0.4489
Test F1 Score    : 0.4000
Test Accuracy    : 0.5000

Top 5 features (by mean |SHAP|):
  - bb_low                    0.1081
  - volume_ratio              0.1008
  - returns_1d                0.0845
  - obv                       0.0755
  - macd_signal               0.0745

Images saved      : 11
Model artifacts   : 3
Timestamp         : 2026-04-20 19:47:06.597218
